In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv('C:/Users/diyad/Desktop/HireSafe/data/DataSet.csv')

print(f"Original dataset shape: {df.shape}")
print(f"Fraudulent distribution:\n{df['fraudulent'].value_counts()}")

Original dataset shape: (17880, 18)
Fraudulent distribution:
fraudulent
f    17014
t      866
Name: count, dtype: int64


In [2]:
# Fill missing text fields with empty string
text_cols = ['title', 'location', 'department', 'company_profile', 
             'description', 'requirements', 'benefits', 'employment_type',
             'required_experience', 'required_education', 'industry', 'function']

for col in text_cols:
    df[col] = df[col].fillna('')

# Create flag for missing company profile (important signal!)
df['has_company_profile'] = (~df['company_profile'].isna()).astype(int)

print("✓ Missing values handled")
print(f"\nPostings with company profile: {df['has_company_profile'].sum()}")
print(f"Postings without company profile: {(1 - df['has_company_profile']).sum()}")

✓ Missing values handled

Postings with company profile: 17880
Postings without company profile: 0


In [4]:
import re

def clean_html(text):
    """Remove HTML tags and clean text"""
    if pd.isna(text) or text == '':
        return ''
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Clean all text columns
text_cols_to_clean = ['company_profile', 'description', 'requirements', 'benefits']

for col in text_cols_to_clean:
    print(f"Cleaning {col}...")
    df[col] = df[col].apply(clean_html)

print("✓ HTML tags removed from all text fields")

Cleaning company_profile...
Cleaning description...
Cleaning requirements...
Cleaning benefits...
✓ HTML tags removed from all text fields


In [5]:
#Combine all text fields for NLP processing
df['full_text'] = (
    df['title'] + ' ' + 
    df['location'] + ' ' + 
    df['department'] + ' ' + 
    df['company_profile'] + ' ' + 
    df['description'] + ' ' + 
    df['requirements'] + ' ' + 
    df['benefits']
)

# Clean the combined text too
df['full_text'] = df['full_text'].apply(clean_html)

print("✓ Combined text field created")
print(f"\nSample CLEANED combined text (first 300 chars):")
print(df['full_text'].iloc[0][:300])

✓ Combined text field created

Sample CLEANED combined text (first 300 chars):
Marketing Intern US, NY, New York Marketing We're Food52, and we've created a groundbreaking and award-winning cooking site. We support, connect, and celebrate home cooks, and give them everything they need in one place. We have a top editorial, business, and engineering team. We're focused on using


In [7]:
# Convert fraudulent column to numeric (it's currently 't'/'f' strings)
df['fraudulent'] = df['fraudulent'].map({'t': 1, 'f': 0})

print(f"✓ Fraudulent column converted to numeric")
print(f"Fraudulent counts:\n{df['fraudulent'].value_counts()}")

# Stratified split: 80% train, 10% val, 10% test
train_df, temp_df = train_test_split(
    df, 
    test_size=0.2, 
    stratify=df['fraudulent'], 
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5, 
    stratify=temp_df['fraudulent'], 
    random_state=42
)

print("\nSplit sizes:")
print(f"Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val: {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test: {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

print("\nClass distribution (fraudulent %):")
print(f"Train: {train_df['fraudulent'].mean()*100:.2f}%")
print(f"Val: {val_df['fraudulent'].mean()*100:.2f}%")
print(f"Test: {test_df['fraudulent'].mean()*100:.2f}%")

✓ Fraudulent column converted to numeric
Fraudulent counts:
fraudulent
0    17014
1      866
Name: count, dtype: int64

Split sizes:
Train: 14304 (80.0%)
Val: 1788 (10.0%)
Test: 1788 (10.0%)

Class distribution (fraudulent %):
Train: 4.84%
Val: 4.87%
Test: 4.81%


In [8]:
# Save splits
train_df.to_csv('../data/train.csv', index=False)
val_df.to_csv('../data/val.csv', index=False)
test_df.to_csv('../data/test.csv', index=False)

print("✓ Processed data saved!")
print(f"\nFiles created:")
print(f"  - data/train.csv ({len(train_df)} rows)")
print(f"  - data/val.csv ({len(val_df)} rows)")
print(f"  - data/test.csv ({len(test_df)} rows)")

OSError: Cannot save file into a non-existent directory: '..\data'